<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/02_window_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')


BASE_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project"

SPLIT_DIR = os.path.join(BASE_DIR, "data/splits")
WINDOW_DIR = os.path.join(BASE_DIR, "data/windows")

os.makedirs(WINDOW_DIR, exist_ok=True)

print("Split directory:", SPLIT_DIR)
print("Window directory:", WINDOW_DIR)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Split directory: /content/drive/MyDrive/2026/www/nasa-mosaics-project/data/splits
Window directory: /content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows


In [2]:
# Choose split
split_name = "train"   # change to "val" when needed

file_map = {
    "train": "train_balanced.csv",
    "val": "val_windows.csv",
    "test": "test_windows.csv"  # future-proof
}

if split_name not in file_map:
    raise ValueError(f"Invalid split_name: {split_name}")

file_path = os.path.join(WINDOW_DIR, file_map[split_name])

print(f"Loading: {file_path}")
df = pd.read_csv(file_path)

print("Rows:", len(df))
print("Unique windows:", df["window_id"].nunique())


Loading: /content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows/train_balanced.csv
Rows: 27000
Unique windows: 450


In [5]:
ml_df = pd.read_csv(f"{SPLIT_DIR}/ml_{split_name}.csv")
jackson_df = pd.read_csv(f"{SPLIT_DIR}/jackson_{split_name}.csv")

print("Pressure rows:", len(ml_df))
print("Events:", len(jackson_df))



Pressure rows: 2513117
Events: 225


In [6]:
ml_df = ml_df.sort_values("SCLK").reset_index(drop=True)
jackson_df = jackson_df.sort_values("SCLK").reset_index(drop=True)


In [7]:
WINDOW_SIZE = 60

windows = []
window_id = 0

for _, event in jackson_df.iterrows():

    event_sclk = event["SCLK"]

    matches = ml_df[ml_df["SCLK"] == event_sclk]
    if matches.empty:
        continue

    event_idx = matches.index[0]

    precursor_region = ml_df.loc[:event_idx]
    precursor_region = precursor_region[
        precursor_region["gt_detection_win"] == True
    ]

    if precursor_region.empty:
        continue

    first_precursor_idx = precursor_region.index[0]

    start_idx = first_precursor_idx - WINDOW_SIZE
    end_idx = first_precursor_idx

    if start_idx < 0:
        continue

    window = ml_df.iloc[start_idx:end_idx].copy()

    if len(window) != WINDOW_SIZE:
        continue

    window["window_id"] = window_id
    window["label"] = 1
    window["event_sclk"] = event_sclk

    windows.append(window)
    window_id += 1



In [8]:
output_file = os.path.join(WINDOW_DIR, file_map[split_name])

# Delete existing file if it exists
if os.path.exists(output_file):
    os.remove(output_file)
    print("Deleted existing file:", output_file)

if windows:
    result_df = pd.concat(windows, ignore_index=True)
    result_df.to_csv(output_file, index=False)

    print("Saved:", output_file)
    print("Positive windows extracted:", window_id)
else:
    print("No windows extracted.")


Deleted existing file: /content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows/train_balanced.csv
Saved: /content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows/train_balanced.csv
Positive windows extracted: 225
